In [2]:
pip install ucimlrepo


In [3]:
# ============================================================
# Breast Cancer Classification Project
# ============================================================

import os
import joblib
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# ============================================================
# Create model directory
# ============================================================

os.makedirs("model", exist_ok=True)

# ============================================================
# Load Dataset
# ============================================================

print("=" * 60)
print("Loading Breast Cancer Wisconsin Diagnostic Dataset")
print("=" * 60)

breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17)

X = breast_cancer_wisconsin_diagnostic.data.features
y = breast_cancer_wisconsin_diagnostic.data.targets

print("\nDataset Shape:")
print("Features:", X.shape)
print("Target:", y.shape)

# ============================================================
# Encode Target Variable
# B -> 0
# M -> 1
# ============================================================

encoder = LabelEncoder()
y = encoder.fit_transform(y.values.ravel())

# ============================================================
# Train-Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)


# ============================================================
# Save Test Data
# Required for Streamlit App
# ============================================================

test_df = X_test.copy()
test_df["target"] = y_test

test_df.to_csv("test_data.csv", index=False)

print("\nTest data saved as test_data.csv")

# ============================================================
# Feature Scaling
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "model/scaler.pkl")

print("Scaler saved.")

# ============================================================
# Define Models
# ============================================================

models = {
    "Logistic Regression":
    LogisticRegression(max_iter=5000, random_state=42),

    "Decision Tree":
    DecisionTreeClassifier(random_state=42),

    "KNN":
    KNeighborsClassifier(n_neighbors=5),

    "Naive Bayes":
    GaussianNB(),

    "Support Vector Machine":
    SVC(probability=True, random_state=42)
}

# ============================================================
# Evaluation Function
# ============================================================

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    return [accuracy, auc, precision, recall, f1, mcc]

# ============================================================
# Train Models and Save
# ============================================================

results = []

for model_name, model in models.items():

    print(f"\nTraining {model_name}...")

    if model_name in [
        "Logistic Regression",
        "KNN",
        "Support Vector Machine"
    ]:

        model.fit(X_train_scaled, y_train)

        scores = evaluate_model(model, X_test_scaled, y_test)

    else:

        model.fit(X_train, y_train)

        scores = evaluate_model(model, X_test, y_test)

    results.append([model_name] + scores)

    filename = model_name.lower().replace(" ", "_") + ".pkl"

    joblib.dump(model, f"model/{filename}")

    print(f"{model_name} saved successfully.")

# ============================================================
# Result Comparison Table
# ============================================================

results_df = pd.DataFrame(results, columns=["ML Model Name", "Accuracy", "AUC",
        "Precision", "Recall", "F1 Score", "MCC"]
)

results_df = results_df.round(4)

print("\n")
print("=" * 90)
print("MODEL COMPARISON TABLE")
print("=" * 90)

print(results_df)

# ============================================================
# Identify Best Model
# ============================================================

best_model = results_df.loc[results_df["Accuracy"].idxmax()]

print("\n")
print("=" * 60)
print("WINNER MODEL")
print("=" * 60)

print(f"Best Model: {best_model['ML Model Name']}")
print(f"Accuracy: {best_model['Accuracy']}")

# ============================================================
# Save Comparison Results
# ============================================================

results_df.to_csv("model_comparison_results.csv", index=False)

print("\nComparison table saved as:")
print("model_comparison_results.csv")
print("\nProject Completed Successfully!")

Loading Breast Cancer Wisconsin Diagnostic Dataset

Dataset Shape:
Features: (569, 30)
Target: (569, 1)

Test data saved as test_data.csv
Scaler saved.

Training Logistic Regression...
Logistic Regression saved successfully.

Training Decision Tree...
Decision Tree saved successfully.

Training KNN...
KNN saved successfully.

Training Naive Bayes...
Naive Bayes saved successfully.

Training Support Vector Machine...
Support Vector Machine saved successfully.


MODEL COMPARISON TABLE
            ML Model Name  Accuracy     AUC  Precision  Recall  F1 Score  \
0     Logistic Regression    0.9649  0.9960     0.9750  0.9286    0.9512   
1           Decision Tree    0.9298  0.9246     0.9048  0.9048    0.9048   
2                     KNN    0.9561  0.9823     0.9744  0.9048    0.9383   
3             Naive Bayes    0.9386  0.9934     1.0000  0.8333    0.9091   
4  Support Vector Machine    0.9737  0.9947     1.0000  0.9286    0.9630   

      MCC  
0  0.9245  
1  0.8492  
2  0.9058  
3  0.87